# AgentVerse Data Cleaning & Analysis with pandas
**HackIndia Skill:** [Clean Data With Pandas](https://hackindia.org/skills/clean-data-with-pandas)  
**Author:** Adarsh Pandey ([@adarsh-pandey-cse](https://github.com/adarsh-pandey-cse))  

This notebook demonstrates loading raw heterogeneous data files, cleaning missing values, formatting dates, deduplicating, merging tables, computing aggregations with pandas and NumPy, and generating visualizations.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Ingest raw CSV and JSON files
tasks_df = pd.read_csv('../data/raw_tasks.csv')
categories_df = pd.read_json('../data/raw_categories.json')
print(f'Raw tasks: {tasks_df.shape}, Categories: {categories_df.shape}')

In [2]:
# 2. Deduplicate records and trim strings
tasks_df = tasks_df.drop_duplicates(subset=['task_id', 'topic'])
tasks_df['topic'] = tasks_df['topic'].str.strip()
tasks_df['priority'] = tasks_df['priority'].fillna('MEDIUM')
tasks_df['duration_seconds'] = tasks_df['duration_seconds'].fillna(tasks_df['duration_seconds'].median())
tasks_df['created_date'] = pd.to_datetime(tasks_df['created_date'], errors='coerce').dt.strftime('%Y-%m-%d')
tasks_df.head()

In [3]:
# 3. Relational Merge
merged_df = pd.merge(tasks_df, categories_df, on='category_id', how='left')
merged_df['duration_minutes'] = np.round(merged_df['duration_seconds'] / 60.0, 2)
merged_df[['task_id', 'topic', 'category_name', 'priority', 'duration_minutes']].head()

In [4]:
# 4. GroupBy Aggregations
summary = merged_df.groupby('category_name').agg(
    total_tasks=('task_id', 'count'),
    avg_duration_sec=('duration_seconds', 'mean'),
    completed=('status', lambda s: (s == 'COMPLETED').sum())
).reset_index()
summary

In [5]:
# 5. Data Visualization
plt.figure(figsize=(7, 4))
plt.bar(summary['category_name'], summary['total_tasks'], color='#1e40af')
plt.xticks(rotation=20, ha='right')
plt.ylabel('Tasks')
plt.title('Agent Tasks by Category')
plt.tight_layout()
plt.show()